# Tuning

**Goal:** improve on the Phase 5 baselines with two levers:

1. **Hyperparameters** — exhaustive `GridSearchCV` over all four model families (~50,000 candidate configurations, ~254,000 individual fits), refit on **F2**.
2. **Operating point** — the decision threshold is tuned on out-of-fold training probabilities to favour recall on leavers, per the project's priority order.

Every family from Phase 5 is re-tuned, not just the leader: tuning routinely reshuffles rankings, and the comparison is cheap relative to the searches themselves. The data is small (1,176 × 56), so the full run is expected to take roughly 30–70 minutes on a modern multi-core desktop.

Scope boundary unchanged: the authoritative test evaluation (confusion matrix, curves) is Phase 7, and SHAP interpretation is Phase 8. This stage ends by persisting a final tuned model and its threshold for those stages to load.

## 1. Setup

The Phase 5 import set plus the tuning machinery: `GridSearchCV`, `cross_val_predict`, `ParameterGrid`, and `fbeta_score`/`make_scorer` for the F2 metric. `RANDOM_STATE = 42` and `PROC` match every earlier stage.

In [13]:
# Cell 1 — Imports & setup
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

_xgb_major = int(xgb.__version__.split('.')[0])
XGB_GPU_KWARGS = {'device': 'cuda'} if _xgb_major >= 2 else {'tree_method': 'gpu_hist'}

from sklearn.model_selection import (
    StratifiedKFold, GridSearchCV, cross_val_predict, ParameterGrid,
)
from sklearn.metrics import (
    recall_score, precision_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, make_scorer,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
PROC = Path('processed')

## 2. Load the modeling matrices and the Phase 5 baseline

All inputs come from `processed/` — no in-memory hand-off between stage notebooks.

Two differences from Phase 5's load cell:

- The **precomputed SMOTE matrices are not loaded**. In this stage SMOTE always sits inside the Logistic Regression pipeline, so it is refit on each CV training fold — including for the final full-train refit — and the standalone matrices are unnecessary.
- `cv_results.parquet` (the Phase 5 cross-validated table) is loaded so every tuned model can be compared against its own baseline.

`scale_pos_weight` (≈ 5.19) is recomputed from the training labels; the grids search a band around it rather than treating it as fixed.

In [ ]:
# Cell 2 — Load the modeling matrices and the Phase 5 baseline
# The precomputed SMOTE matrices are NOT loaded here: in this stage SMOTE always
# lives inside the LogReg pipeline, so resampling is refit on each CV training fold.
X_train = pd.read_parquet(PROC / 'X_train.parquet')
X_test = pd.read_parquet(PROC / 'X_test.parquet')
y_train = pd.read_parquet(PROC / 'y_train.parquet')['Attrition']
y_test = pd.read_parquet(PROC / 'y_test.parquet')['Attrition']

# Phase 5 cross-validated results, for the before/after-tuning comparison.
baseline_cv = pd.read_parquet(PROC / 'cv_results.parquet')

# scale_pos_weight = #negatives / #positives (same computation as Phase 5).
spw = (y_train == 0).sum() / (y_train == 1).sum()

print(f'X_train {X_train.shape} | attrition {y_train.mean():.3f}')
print(f'X_test  {X_test.shape}  | attrition {y_test.mean():.3f}')
print(f'scale_pos_weight = {spw:.3f}')

X_train (1176, 56) | attrition 0.162
X_test  (294, 56)  | attrition 0.160
scale_pos_weight = 5.189



## 3. Helpers, scorers and the search wrapper

Helpers are defined locally because functions do not carry across stage notebooks.

- `metrics_at_threshold()` / `sweep_thresholds()` — recall, precision, F1 and F2 on the positive class at an arbitrary probability threshold; used by the threshold sweep and every readout table.
- **Scoring:** multi-metric (recall, precision, F1, F2, ROC-AUC, PR-AUC) with **`refit='f2'`**. Refitting on pure recall would be degenerate — across ~50k candidates, the heaviest class weighting wins by predicting nearly everyone as a leaver (recall → 1.0 at base-rate precision). F2 weights recall four times as heavily as precision, so it leans the search toward the business priority while a predict-all-positive solution (F2 ≈ 0.49 at 16% prevalence) cannot win. Pushing the recall/precision operating point further is the threshold's job, not the grid's.
- `run_search()` — wraps `GridSearchCV(n_jobs=-1)`, prints the candidate count up front and the elapsed time and best parameters when done. Because `refit='f2'`, each search ends with `best_estimator_` already refit on the full training set.

The CV splitter is the same `StratifiedKFold(5, shuffle=True, random_state=42)` as Phase 5, so tuned and baseline scores are directly comparable.

In [ ]:
# Cell 3 — Helpers, scorers and the search wrapper
# Defined here because helpers do not carry across stage notebooks.
def metrics_at_threshold(y_true, proba, threshold):
    """Recall/precision/F1/F2 on the positive 'Yes' class at a probability threshold."""
    pred = (proba >= threshold).astype(int)
    return {
        'recall': recall_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred),
        'f2': fbeta_score(y_true, pred, beta=2),
    }


def sweep_thresholds(y_true, proba, thresholds):
    """One row of threshold metrics per candidate threshold."""
    rows = [{'threshold': t, **metrics_at_threshold(y_true, proba, t)} for t in thresholds]
    return pd.DataFrame(rows)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'f2': make_scorer(fbeta_score, beta=2),
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
}


def run_search(estimator, param_grid, name):
    """Exhaustive GridSearchCV refit on F2; prints candidate count and elapsed time."""
    n_candidates = len(ParameterGrid(param_grid))
    print(f'{name}: {n_candidates:,} candidates x 5 folds = {n_candidates * 5:,} fits')
    search = GridSearchCV(
        estimator, param_grid, scoring=scoring, refit='f2',
        cv=cv, n_jobs=-1, verbose=1,
    )
    t0 = time.perf_counter()
    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    print(f'{name}: done in {elapsed / 60:.1f} min | best CV F2 = {search.best_score_:.3f}')
    print(f'{name}: best params = {search.best_params_}')
    return search


searches = {}

## 4. Logistic Regression search

The Phase 5 leader gets the most structurally varied grid. It is a **list of six dicts** — {SMOTE-on, SMOTE-off} × {l2, l1, elasticnet} — so invalid combinations (such as `l1_ratio` outside elasticnet) are never generated:

- **SMOTE-on:** `sampling_strategy` 0.5–1.0 and `k_neighbors` 3–9 are searched as pipeline parameters (`smote__…`), optionally combined with `class_weight='balanced'`.
- **SMOTE-off:** `smote: 'passthrough'` disables the resampler entirely, testing whether class weights alone (including explicit 1:3 … 1:6 dicts) beat synthetic oversampling.
- Both variants cross 11 values of `C` with all three penalties (`solver='saga'` supports them all; `max_iter=5000` because saga converges slowly — the features were standardized in Phase 2, so it is well-conditioned).

≈ 2,035 candidates → ~10,200 fits.

In [ ]:
# Cell 4 — Logistic Regression search (SMOTE-on and SMOTE-off variants)
# The grid is a list of dicts so invalid combinations (e.g. l1_ratio without
# elasticnet) are never generated. saga supports all three penalties.
C_LIST = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]

logreg_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', LogisticRegression(solver='saga', max_iter=5000, random_state=RANDOM_STATE)),
])

smote_on = {
    'smote__sampling_strategy': [0.5, 0.65, 0.8, 1.0],
    'smote__k_neighbors': [3, 5, 7, 9],
    'clf__class_weight': [None, 'balanced'],
}
smote_off = {
    'smote': ['passthrough'],
    'clf__class_weight': ['balanced', {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}, {0: 1, 1: 6}],
}
penalties = [
    {'clf__penalty': ['l2'], 'clf__C': C_LIST},
    {'clf__penalty': ['l1'], 'clf__C': C_LIST},
    {'clf__penalty': ['elasticnet'], 'clf__C': C_LIST, 'clf__l1_ratio': [0.2, 0.5, 0.8]},
]

logreg_grid = [
    {**variant, **penalty}
    for variant in (smote_on, smote_off)
    for penalty in penalties
]

searches['LogReg (SMOTE)'] = run_search(logreg_pipe, logreg_grid, 'LogReg (SMOTE)')

LogReg (SMOTE): 2,035 candidates x 5 folds = 10,175 fits
Fitting 5 folds for each of 2035 candidates, totalling 10175 fits
LogReg (SMOTE): done in 5.3 min | best CV F2 = 0.646
LogReg (SMOTE): best params = {'clf__C': 0.03, 'clf__class_weight': {0: 1, 1: 4}, 'clf__l1_ratio': 0.5, 'clf__penalty': 'elasticnet', 'smote': 'passthrough'}


## 5. Random Forest search

Tree shape (`max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`), forest size, and five imbalance treatments — `'balanced'`, `'balanced_subsample'`, and explicit 1:4 / 1:6 / 1:8 weight dicts that punish missed leavers harder than the ~5.2:1 base rate.

The estimator runs with `n_jobs=1`: `GridSearchCV` already parallelises across candidates, and nesting parallelism inside each fit oversubscribes the CPU.

576 candidates → 2,880 fits. This is the slowest family per fit so fewer parameters were tweaked.

In [16]:
# Cell 5 — Random Forest search
# n_jobs=1 on the estimator: GridSearchCV already parallelises across candidates,
# and nesting parallelism oversubscribes the CPU.
rf_grid = {
    'n_estimators': [300],
    'max_depth': [None, 4, 6, 8, 12, 16],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 0.3, 0.5],
    'class_weight': ['balanced', {0: 1, 1: 4}, {0: 1, 1: 6}, {0: 1, 1: 8}],
}

searches['RandomForest'] = run_search(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    rf_grid, 'RandomForest',
)

RandomForest: 576 candidates x 5 folds = 2,880 fits
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
RandomForest: done in 4.5 min | best CV F2 = 0.580
RandomForest: best params = {'class_weight': {0: 1, 1: 8}, 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 8, 'min_samples_split': 2, 'n_estimators': 300}


## 6. XGBoost search

The full regularisation surface: boosting size vs learning rate, tree complexity (`max_depth`, `min_child_weight`, `gamma`), row/column subsampling, and both L1/L2 penalties — crossed with three `scale_pos_weight` settings (half, exact, and 1.5× the 5.19 base ratio). `tree_method='hist'` keeps each fit fast on this data size.

2,304 candidates → 11,520 fits.

In [19]:
# Cell 6 — XGBoost search
xgb_grid = {
    'n_estimators': [ 300, 600],
    'learning_rate': [ 0.05, 0.1],
    'max_depth': [ 3, 4, 6],
    'min_child_weight': [1, 5],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.5, 0.8],
    'gamma': [0, 1],
    'reg_alpha': [ 0 ],
    'reg_lambda': [ 1 ],
    'scale_pos_weight': [ spw ],
}

searches['XGBoost'] = run_search(
    XGBClassifier(eval_metric='logloss', **XGB_GPU_KWARGS,
                  random_state=RANDOM_STATE, n_jobs=1),
    xgb_grid, 'XGBoost',
)

XGBoost: 192 candidates x 5 folds = 960 fits
Fitting 5 folds for each of 192 candidates, totalling 960 fits
XGBoost: done in 11.6 min | best CV F2 = 0.573
XGBoost: best params = {'colsample_bytree': 0.8, 'gamma': 1, 'learning_rate': 0.05, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 300, 'reg_alpha': 0, 'reg_lambda': 1, 'scale_pos_weight': np.float64(5.189473684210526), 'subsample': 1.0}


## 7. LightGBM search

The leaf-wise analogue of the XGBoost grid: `num_leaves` is LightGBM's primary complexity control, searched alongside `max_depth`, `min_child_samples`, subsampling, regularisation, and the same three `scale_pos_weight` settings. `subsample_freq=1` is required — without it LightGBM silently ignores `subsample`.

1,728 candidates → 8,640 fits; LightGBM is the fastest family per fit.

In [20]:
# Cell 7 — LightGBM search
# subsample_freq=1 is required for subsample to take effect in LightGBM.
lgbm_grid = {
    'n_estimators': [300, 600],
    'learning_rate': [ 0.05, 0.1],
    'num_leaves': [ 15, 31],
    'max_depth': [-1],
    'min_child_samples': [ 10, 40],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.5, 0.8],
    'reg_alpha': [0, 1],
    'reg_lambda': [0, 5],
    'scale_pos_weight': [ spw ],
}

searches['LightGBM'] = run_search(
    LGBMClassifier(random_state=RANDOM_STATE, n_jobs=1, verbose=-1, subsample_freq=1,
                   device='gpu'),
    lgbm_grid, 'LightGBM',
)

LightGBM: 256 candidates x 5 folds = 1,280 fits
Fitting 5 folds for each of 256 candidates, totalling 1280 fits
LightGBM: done in 24.1 min | best CV F2 = 0.560
LightGBM: best params = {'colsample_bytree': 0.5, 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 10, 'n_estimators': 300, 'num_leaves': 15, 'reg_alpha': 1, 'reg_lambda': 5, 'scale_pos_weight': np.float64(5.189473684210526), 'subsample': 0.7}


## 8. Tuned vs Phase 5 baseline

Cross-validated metrics of each family's best configuration at the **default 0.5 threshold**, joined against the Phase 5 baseline table. This isolates what hyperparameter search alone bought, before the threshold moves — the two levers are reported separately so their contributions aren't conflated.

In [ ]:
# Cell 8 — Tuned vs Phase 5 baseline (default 0.5 threshold)
# What hyperparameter search alone bought, before any threshold movement.
rows = []
for name, search in searches.items():
    res = search.cv_results_
    i = search.best_index_
    rows.append({
        'model': name,
        'recall': res['mean_test_recall'][i],
        'precision': res['mean_test_precision'][i],
        'f1': res['mean_test_f1'][i],
        'f2': res['mean_test_f2'][i],
        'roc_auc': res['mean_test_roc_auc'][i],
        'pr_auc': res['mean_test_pr_auc'][i],
    })

tuned_cv = pd.DataFrame(rows)
compare = tuned_cv.merge(
    baseline_cv[['model', 'recall', 'f1']], on='model', suffixes=('_tuned', '_phase5'),
)
compare['recall_gain'] = compare['recall_tuned'] - compare['recall_phase5']
compare['f1_gain'] = compare['f1_tuned'] - compare['f1_phase5']
print(compare.sort_values('f2', ascending=False).to_string(index=False))

         model  recall_tuned  precision  f1_tuned    f2  roc_auc  pr_auc  recall_phase5  f1_phase5  recall_gain  f1_gain
LogReg (SMOTE)         0.700      0.508     0.583 0.646    0.827   0.635          0.516      0.511        0.184    0.072
  RandomForest         0.700      0.351     0.465 0.580    0.782   0.531          0.174      0.286        0.526    0.179
       XGBoost         0.574      0.598     0.577 0.573    0.804   0.602          0.426      0.520        0.147    0.057
      LightGBM         0.547      0.630     0.583 0.560    0.818   0.628          0.437      0.522        0.111    0.061


## 9. Out-of-fold probabilities and the threshold sweep

The threshold cannot be chosen on the test set (that would leak it into model selection), and the full-train fit's in-sample probabilities would be optimistic. Instead, `cross_val_predict` produces **out-of-fold probabilities**: every training row is scored by a model that never saw it. The LogReg pipeline re-applies SMOTE inside each fold, so this stays leakage-free.

Each model's probabilities are swept across 91 thresholds (0.05–0.95, step 0.01) and four criteria are tabulated:

| Criterion | Rationale |
|---|---|
| **max F2** (primary) | recall-weighted, no arbitrary constants, comparable across models |
| max F1 | the balanced reference point |
| max recall @ precision ≥ 0.30 | an HR-capacity framing — flag as many leavers as possible while ~1 in 3 flags is real (≈ 2× the 16% base rate) |
| default 0.50 | what the model does untouched |

The **max-F2 threshold** is the one carried forward per model; the other rows show the trade-off space.

In [ ]:
# Cell 9 — Out-of-fold probabilities and the threshold sweep
# OOF probabilities come from cross_val_predict with the same 5-fold splitter, so
# every probability is produced by a model that never saw that row. The test set
# plays no part in choosing the threshold.
THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)

oof_proba = {}
tuned_threshold = {}
sweeps = []
criteria_rows = []
for name, search in searches.items():
    proba = cross_val_predict(
        search.best_estimator_, X_train, y_train,
        cv=cv, method='predict_proba', n_jobs=-1,
    )[:, 1]
    oof_proba[name] = proba

    sweep = sweep_thresholds(y_train, proba, THRESHOLDS)
    sweep.insert(0, 'model', name)
    sweeps.append(sweep)

    crit_list = [
        ('max F2', sweep.loc[sweep['f2'].idxmax(), 'threshold']),
        ('max F1', sweep.loc[sweep['f1'].idxmax(), 'threshold']),
    ]
    feasible = sweep[sweep['precision'] >= 0.30]
    if len(feasible):
        crit_list.append((
            'max recall @ precision >= 0.30',
            feasible.loc[feasible['recall'].idxmax(), 'threshold'],
        ))
    crit_list.append(('default 0.50', 0.50))

    tuned_threshold[name] = crit_list[0][1]  # primary criterion: max OOF F2
    for crit, t in crit_list:
        criteria_rows.append({
            'model': name, 'criterion': crit, 'threshold': t,
            **metrics_at_threshold(y_train, proba, t),
        })

threshold_sweep = pd.concat(sweeps, ignore_index=True)
criteria_table = pd.DataFrame(criteria_rows)
print(criteria_table.to_string(index=False))

         model                      criterion  threshold  recall  precision    f1    f2
  RandomForest                         max F2      0.490   0.716      0.334 0.456 0.583
  RandomForest                         max F1      0.570   0.526      0.452 0.487 0.510
  RandomForest max recall @ precision >= 0.30      0.470   0.737      0.305 0.431 0.574
  RandomForest                   default 0.50      0.500   0.700      0.345 0.463 0.581
       XGBoost                         max F2      0.420   0.626      0.494 0.552 0.594
       XGBoost                         max F1      0.490   0.584      0.575 0.580 0.582
       XGBoost max recall @ precision >= 0.30      0.210   0.774      0.304 0.437 0.591
       XGBoost                   default 0.50      0.500   0.574      0.583 0.578 0.576
      LightGBM                         max F2      0.310   0.663      0.474 0.553 0.614
      LightGBM                         max F1      0.430   0.579      0.576 0.577 0.578
      LightGBM max recall @ prec

## 10. Final comparison and leader selection

Each model's out-of-fold metrics at its own tuned threshold, ranked by the project rule from Phase 4: **recall first, F1 tiebreak**. The top row is the tuned leader. ROC-AUC and PR-AUC are threshold-free and reported for context.

In [ ]:
# Cell 10 — Final comparison and leader selection
# OOF metrics at each model's tuned (max-F2) threshold, ranked by the project rule:
# recall first, F1 tiebreak.
final_rows = []
for name in searches:
    t = tuned_threshold[name]
    proba = oof_proba[name]
    final_rows.append({
        'model': name,
        'threshold': t,
        **metrics_at_threshold(y_train, proba, t),
        'roc_auc': roc_auc_score(y_train, proba),
        'pr_auc': average_precision_score(y_train, proba),
    })

final_results = (
    pd.DataFrame(final_rows)
    .sort_values(['recall', 'f1'], ascending=False)
    .reset_index(drop=True)
)
leader = final_results.iloc[0]['model']
print(final_results.to_string(index=False))
print(f'\nTuned leader (OOF recall -> F1 at tuned threshold): {leader}')

         model  threshold  recall  precision    f1    f2  roc_auc  pr_auc
  RandomForest      0.490   0.716      0.334 0.456 0.583    0.777   0.512
LogReg (SMOTE)      0.500   0.700      0.496 0.581 0.647    0.826   0.627
      LightGBM      0.310   0.663      0.474 0.553 0.614    0.814   0.623
       XGBoost      0.420   0.626      0.494 0.552 0.594    0.801   0.596

Tuned leader (OOF recall -> F1 at tuned threshold): RandomForest


## 11. Light test-set readout (context only)

The Phase 5 pattern: the leader (already refit on the full training set by `GridSearchCV`) is read off against the held-out test set at both the tuned and the default threshold. **Indicative only** — it confirms the out-of-fold picture transfers to unseen data; the authoritative evaluation is Phase 7.

In [ ]:
# Cell 11 — Light test-set readout (context only)
# Indicative only — the authoritative test evaluation is Phase 7.
# best_estimator_ was already refit on the full training set by GridSearchCV.
leader_model = searches[leader].best_estimator_
leader_t = tuned_threshold[leader]

test_proba = leader_model.predict_proba(X_test)[:, 1]
sanity = pd.DataFrame([
    {'setting': f'tuned threshold ({leader_t:.2f})',
     **metrics_at_threshold(y_test, test_proba, leader_t)},
    {'setting': 'default threshold (0.50)',
     **metrics_at_threshold(y_test, test_proba, 0.50)},
])
sanity['roc_auc'] = roc_auc_score(y_test, test_proba)
sanity['pr_auc'] = average_precision_score(y_test, test_proba)
print(f'Leader on test (indicative only): {leader}')
print(sanity.to_string(index=False))

Leader on test (indicative only): RandomForest
                 setting  recall  precision    f1    f2  roc_auc  pr_auc
  tuned threshold (0.49)   0.830      0.315 0.456 0.625    0.766   0.397
default threshold (0.50)   0.809      0.317 0.455 0.617    0.766   0.397


## 12. Persist artifacts

Everything Phases 7 and 8 need, written to `processed/`:

| Artifact | Contents |
|---|---|
| `models/*_tuned.joblib` | each family's `best_estimator_`, refit on the full training set |
| `models/final_model.joblib` | the tuned leader — the single entry point for Phases 7–8 |
| `tuning_results.parquet` | per model: search size, best CV F2, tuned threshold, OOF metrics at that threshold and at 0.5, Phase 5 baseline |
| `threshold_sweep.parquet` | the full 4 × 91 threshold sweep, so Phase 7 can plot it without recomputing |
| `best_params.joblib` | winning hyperparameters for all four families |
| `tuning_selection.joblib` | leader name, slug, **chosen threshold**, criteria and the selection rule |

Phase 7 contract: load `models/final_model.joblib` and `tuning_selection.joblib['threshold']`, then classify the test set with `proba >= threshold`.

In [ ]:
# Cell 12 — Persist artifacts for Phases 7 and 8
OUT = PROC / 'models'
OUT.mkdir(parents=True, exist_ok=True)

slug = {
    'LogReg (SMOTE)': 'logreg_smote',
    'RandomForest': 'random_forest',
    'XGBoost': 'xgboost',
    'LightGBM': 'lightgbm',
}
for name, search in searches.items():
    joblib.dump(search.best_estimator_, OUT / f'{slug[name]}_tuned.joblib')
joblib.dump(leader_model, OUT / 'final_model.joblib')

summary_rows = []
for name, search in searches.items():
    t = tuned_threshold[name]
    at_t = metrics_at_threshold(y_train, oof_proba[name], t)
    at_default = metrics_at_threshold(y_train, oof_proba[name], 0.50)
    base = baseline_cv.loc[baseline_cv['model'] == name].iloc[0]
    summary_rows.append({
        'model': name,
        'n_candidates': len(ParameterGrid(search.param_grid)),
        'best_cv_f2': search.best_score_,
        'threshold': t,
        'oof_recall': at_t['recall'],
        'oof_precision': at_t['precision'],
        'oof_f1': at_t['f1'],
        'oof_f2': at_t['f2'],
        'oof_roc_auc': roc_auc_score(y_train, oof_proba[name]),
        'oof_pr_auc': average_precision_score(y_train, oof_proba[name]),
        'recall_at_05': at_default['recall'],
        'f1_at_05': at_default['f1'],
        'phase5_recall': base['recall'],
        'phase5_f1': base['f1'],
    })

tuning_results = pd.DataFrame(summary_rows)
tuning_results.to_parquet(PROC / 'tuning_results.parquet', index=False)
threshold_sweep.to_parquet(PROC / 'threshold_sweep.parquet', index=False)

joblib.dump({name: search.best_params_ for name, search in searches.items()},
            PROC / 'best_params.joblib')

joblib.dump({
    'leader': leader,
    'slug': slug[leader],
    'threshold': float(leader_t),
    'threshold_criterion': 'max OOF F2',
    'refit_metric': 'f2',
    'selection_rule': 'oof_recall_then_f1_at_tuned_threshold',
    'best_params': searches[leader].best_params_,
    'oof_metrics': metrics_at_threshold(y_train, oof_proba[leader], leader_t),
}, PROC / 'tuning_selection.joblib')

print(f'Saved 4 tuned models + final_model.joblib to {OUT.resolve()}')
print(f'Saved tuning_results.parquet, threshold_sweep.parquet, best_params.joblib '
      f'and tuning_selection.joblib to {PROC.resolve()}')

Saved 4 tuned models + final_model.joblib to C:\Users\zeesh\Documents\GitHub\Attrition Prediction Model\notebooks\processed\models
Saved tuning_results.parquet, threshold_sweep.parquet, best_params.joblib and tuning_selection.joblib to C:\Users\zeesh\Documents\GitHub\Attrition Prediction Model\notebooks\processed
